# last benchmarked on Ryzen 9 9900X3D with single hardware thread

In [1]:
import time
import numpy as np
def bench_one_fn(fn, reps, **kw):
    # warmup + get the price
    V, grid = fn(**kw)
    S0 = kw['S0']
    x = np.log(grid['S'])
    option_price = np.interp(np.log(S0), x, V)
    best_time = float('inf')
    for _ in range(reps):
        t0 = time.perf_counter()
        fn(**kw)
        best_time = min(best_time, time.perf_counter() - t0)
    return best_time, option_price

In [2]:
from src.fdm import pde_crank_nicolson, pde_crank_nicolson_v2, pde_crank_nicolson_v3, pde_crank_nicolson_v4, pde_crank_nicolson_v5, pde_crank_nicolson_v6, pde_crank_nicolson_v7, pde_crank_nicolson_american
from functools import partial
import pandas as pd
import psutil

p = psutil.Process()
affinity_before = p.cpu_affinity()
print(f"cpu affinity before: {affinity_before}")
p.cpu_affinity([psutil.cpu_count() - 1])
print(f"cpu affinity after: {p.cpu_affinity()}")

fdm_solvers = {
    "european_call_v1": pde_crank_nicolson,
    "european_call_v2": pde_crank_nicolson_v2,
    "european_call_v3": pde_crank_nicolson_v3,
    "european_call_v4": pde_crank_nicolson_v4,
    "european_call_v5": pde_crank_nicolson_v5,
    "european_call_v6": pde_crank_nicolson_v6,
    "european_call_v7": pde_crank_nicolson_v7,
    "american_call_v1": partial(pde_crank_nicolson_american, contract_type="call"),
    "american_put_v1":  partial(pde_crank_nicolson_american, contract_type="put"),
}

total_solvers = len(fdm_solvers.keys())
rows = []
runs_complete = 0
reps = 8
print(f"\rruns complete: {runs_complete}/{total_solvers * reps}", end=" ", flush=True)
for v, fn in fdm_solvers.items():
    best_time, option_price = bench_one_fn(
        fn=fn,
        reps=reps,
        K=100,
        S0=100,
        r=0.05,
        T=3,
        sigma=0.2,
        s_steps=3000,
        t_steps=3000
    )
    rows.append(
        {
            "v":v,
            'best_time (ms)':best_time*1e3,
            #'price':option_price
        }
    )
    runs_complete+=reps
    print(f"\rruns complete: {runs_complete}/{total_solvers * reps}", end=" ", flush=True)
print()


p.cpu_affinity(affinity_before)
print(f"cpu affinity reset back to {p.cpu_affinity()}")
df = pd.DataFrame(rows)
print(df)

cpu affinity before: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
cpu affinity after: [23]
runs complete: 72/72 
cpu affinity reset back to [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
                  v  best_time (ms)
0  european_call_v1        145.8658
1  european_call_v2        122.2958
2  european_call_v3         89.2788
3  european_call_v4         80.4050
4  european_call_v5         55.0370
5  european_call_v6         50.6914
6  european_call_v7         19.3898
7  american_call_v1         19.8039
8   american_put_v1         19.6531
